In [4]:
import sys
from hra_amap.registration.organ import Organ
from hra_amap.registration.tissue import TissueBlock
from hra_amap.registration.pipeline import Pipeline
from hra_amap.registration.dataclass import Projection
from hra_amap.utils.conversions import to_pointcloud

import hra_api_client
from hra_api_client.api import v1_api
from hra_amap.utils.io import read_yaml
from hra_amap.utils.constants import ConfigKeys
from hra_amap.utils.non_hra_mapping import build_mesh_from_sample, scale_millitome_block, build_blocks_and_donor_points, generate_extraction_sites_jsonld_from_blocks, generate_dataset_graph_jsonld_from_blocks 
from hra_amap.cli.registration_stage_2 import ProjectionBlockGenerator, TissueBlockGenerator 

import time
import trimesh
import numpy as np
from copy import deepcopy
from tqdm.auto import tqdm
from pathlib import Path

In [5]:
# Load configuration file
config = Path("../../../input-data/external-atlas/large-intestine-female-gca/config.yaml")
config_dict = read_yaml(config)
config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.SOURCE] = (
            config.parent
            / config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.SOURCE]
        )
config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.TARGET] = "../../../raw-data/external-atlas/large-intestine-female-gca/3d-sbu-f-large-intestine.glb"

In [6]:
# Set up and initialize the HRA API client to interact with the Human Atlas API
configuration = hra_api_client.Configuration(
    host = "https://apps.humanatlas.io/api" 
)

api_client = hra_api_client.ApiClient(configuration)
api_instance = v1_api.V1Api(api_client)

In [7]:
db_ready = False
result = None
# Poll the HRA backend until the database is ready before proceeding with further API calls
while not db_ready:
    result = api_instance.db_status()
    if result.status == 'Ready':
        db_ready = True 
    else:
        print('Database not ready yet! Retrying...', result)
        time.sleep(2)
print('Database ready!\n', result)

Database ready!
 status='Ready' checkback=3600000 load_time=22594 message='Database successfully loaded'


In [14]:
import requests
import json
# Fetch the knowledge graph for a given ontology term from the HRA Data Service API
try:
    url = "https://apps.humanatlas.io/api/v1/ds-graph"
    params = {"ontology-terms": "http://purl.obolibrary.org/obo/UBERON_0000059",
              "sex" : "Female"}

    response = requests.get(url, params=params)
    data = response.json()
    graph_size = len(data.get("@graph", []))
    print(graph_size)
except hra_api_client.ApiException as e:
    print("Exception when calling DefaultApi->aggregate_results: %s\n" % e)

7


In [15]:
result = data.get("@graph", [])

In [16]:
projection_path =  Path('../../../raw-data/external-atlas/large-intestine-female-gca/projections.pickle.gz')
projection = Projection.load(projection_path)

In [17]:
# Load stage-1 projection data, generate projected tissue blocks and compute their oriented bounding boxes
stage_1_projection_Path = projection_path
projected_blocks = ProjectionBlockGenerator(stage_1_projection_Path, config).generate_projections()

projected_blocks_obb = {id: block.bounding_box_oriented for id, block in deepcopy(projected_blocks).items()}

for index, obb in projected_blocks_obb.items():
    obb.visual.vertex_colors = projected_blocks[index].visual.vertex_colors[0]

In [18]:
import numpy as np
import trimesh

reddishpink = np.array([222, 49, 99, 100], dtype=np.uint8)
MODEL_SCALE = 1

# Load source model
source_path = config_dict[ConfigKeys.INPUT_FILES][ConfigKeys.SOURCE]
scene_or_mesh = trimesh.load(source_path)

if isinstance(scene_or_mesh, trimesh.Scene):
    source_model = trimesh.util.concatenate(scene_or_mesh.dump())
else:
    source_model = scene_or_mesh

source_model.apply_translation(-source_model.centroid)

angle_radx = np.deg2rad(-100)
angle_rady = np.deg2rad(0)
angle_radz = np.deg2rad(-15)

rot = trimesh.transformations.euler_matrix(
    angle_radx, angle_rady, angle_radz, axes='sxyz'
)
source_model.apply_transform(rot)
source_model.fix_normals()

source_min, source_max = source_model.bounds
source_range = source_max - source_min
scaling_factor = np.mean(source_range) / 0.3

blocks, donor_points = build_blocks_and_donor_points(
    result,
    scaling_factor,
    "rxyz"
)
donor_global_center = donor_points.mean(axis=0)

source_center = source_model.centroid


for block, donor_center in zip(blocks, donor_points):
    # offset relative to donor dataset center
    relative_offset = donor_center - donor_global_center

    # final position in organ space
    mapped_position = source_center + relative_offset

    # translate only
    block.apply_translation(mapped_position - block.centroid)


scale_millitome_block(blocks, MODEL_SCALE)

In [19]:
# Color source model
source_model.visual = trimesh.visual.ColorVisuals(
    mesh=source_model,
    vertex_colors=np.tile(reddishpink, (source_model.vertices.shape[0], 1))
)

# Build scene
scene = trimesh.Scene()
scene.add_geometry(
    source_model,
    node_name="source_model",
    geom_name="source_model"
)

for i, block in enumerate(blocks):
    label = block.metadata.get("id", f"block_{i}")
    short_label = label.split("#")[-1]

    scene.add_geometry(
        block,
        node_name=short_label,
        geom_name=short_label
    )


scene.show(flags={'cull': False})

In [ ]:
# scene.export("large-intestine-female-gca.glb");